# Milestone 2: matched size-variant training

Train fresh seed-0 weights on the unique union of **18,432 original images + 21,024 new size variants**. Preserve margins, gaps, origins, colors and triangle size; relax vertical center alignment only for the variants. Every eligible family includes all four circle/square size combinations.

Keep the architecture, R=2, optimizer and **5,760-update / 184,320-presentation budget** fixed. The larger corpus has 118,368 QAs, each visited once or twice. Compare final performance on unchanged validation records, the original training subset, and the same frozen size-intervention population. No test inference or automatic training extension.

Enable GPU and internet. Attach both `milestone2_training_budget_artifacts.zip` and `milestone2_size_intervention_artifacts.zip`, then set the two source paths below. ZIPs or extracted run directories are accepted. Select a committed `REPO_REF` after pushing the implementation. One GPU, float32, batch 32; preserve Kaggle's PyTorch installation.

See `docs/milestones/milestone2_matched_size_training.md`. Use fresh output paths; training resume is unsupported.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # A committed revision containing this notebook; commit SHA preferred.
REPO_DIR = "/kaggle/working/multi-modal-loop-matched-size-training"
RUN_ROOT = "/kaggle/working/milestone2_matched_size_training"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_training_budget_artifacts.zip"
)
INTERVENTION_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_size_intervention_artifacts.zip"
)


## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit references and derive the corpus

Audit the pinned 5,760-update checkpoint and size-intervention summary. Reference weights never initialize training. Preserve exact validation/test records, reject training origin overlap with held-out layouts, build complete size families, deduplicate scenes, and record exposures before training.


In [ ]:
import multimodal_loop.eval.kaggle_matched_size as helpers
from multimodal_loop.eval.kaggle_matched_size import (
    archive_matched_size,
    prepare_matched_size,
    run_matched_size,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_matched_size(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE, INTERVENTION_SOURCE)

## Train and evaluate

Train for one full pass plus 2,061 batches of pass two; validate every 288 updates and evaluate the final checkpoint. Diagnose the expanded training corpus, its original/added subsets, and unchanged validation questions with existing controls. Then repeat the five-condition size intervention on the new frozen model. Only that diagnostic permits edge contact; training retains blank margins.


In [ ]:
report = run_matched_size(run)

## Inspect results

The nine direct-grounding criteria remain unchanged: training per-shape criteria use the full expanded corpus. Compare original-subset training fit separately. Inspect relative-size swaps, circle/square paired accuracy, difficult-layout performance and size invariance; aggregate improvement alone does not establish reliable grounding. Size-intervention comparisons remain descriptive.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

for split, metrics in report["splits"].items():
    print(split, {k: metrics[k] for k in ("total", "accuracy", "loss", "all_three")})
    print("Shape accuracy:", metrics["breakdowns"]["shape"])
print(
    "Training subsets:",
    {
        name: {k: m[k] for k in ("total", "accuracy", "loss")}
        for name, m in report["training_subsets"].items()
    },
)
print(json.dumps(report["assessment"], indent=2))
comparison = json.loads((run.root / "diagnosis" / "comparison.json").read_text())
print(json.dumps({k: comparison[k] for k in ("validation", "original_training", "control_gaps", "both_correct_all_four")}, indent=2))
for split, metrics in report["splits"].items():
    print(split, "Circle/square pair:", metrics["circle_square_pair"])
    print(split, "Identical circle/square predictions:", metrics["circle_square_same_prediction"])
presentations = json.loads((run.root / "training" / "presentations.json").read_text())
print("Presentations:", presentations["qa_visit_histogram"])
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))
print("Relative-size diagnostics:", json.dumps({
    split: {name: {"pair_accuracy": group["metrics"]["circle_square_pair"]["accuracy"],
                   "shape_predictions": group["shape_predictions"]}
            for name, group in groups.items()}
    for split, groups in report["relative_size"].items()
}, indent=2))
display(HTML((run.root / "diagnosis" / "size_intervention.html").read_text()))


## Retain artifacts

Download `milestone2_matched_size_training_artifacts.zip` and bring it back for review. It contains the new corpus and derivation, checkpoint, histories, exposure reports, full predictions, both diagnostic previews, comparisons, provenance and logs. Staged reference weights are excluded.


In [ ]:
archive = archive_matched_size(run)
print("Training-budget archive:", archive)
display(FileLink(str(archive)))